# BT5153 Group 4 - Modeling Pipeline (No Insights/Visualization Write-up)

This notebook is designed to cover modeling, feature analysis, and evaluation artifacts for the final report rubric using the cleaned dataset already produced.

Scope intentionally excludes business insight narration and advanced visualization polish (handled by teammate).

## 1) Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.decomposition import LatentDirichletAllocation, NMF

SEED = 42
MAX_ROWS = 300_000
MAX_FEATURE_SAMPLE = 100_000
MAX_NEG_SAMPLE = 50_000
RUN_BERTOPIC = False

DATA_FILE = Path("electronics_filtered.csv")
META_FILE = Path("electronics_metadata.csv")

assert DATA_FILE.exists(), "Missing electronics_filtered.csv in workspace root"
assert META_FILE.exists(), "Missing electronics_metadata.csv in workspace root"

print("Found files:", DATA_FILE, META_FILE)
print("Row cap for stable execution:", MAX_ROWS)

ModuleNotFoundError: No module named 'polars'

## 2) Load Data and Build Labels

In [ ]:
df = pl.scan_csv(
    str(DATA_FILE),
    schema_overrides={"asin": pl.Utf8},
    n_rows=MAX_ROWS
).select([
    "asin", "overall", "reviewText", "summary", "verified"
]).collect()

df = df.with_columns([
    pl.when(pl.col("overall") <= 2).then(pl.lit("negative"))
      .when(pl.col("overall") == 3).then(pl.lit("neutral"))
      .otherwise(pl.lit("positive"))
      .alias("sentiment"),
    pl.col("reviewText").fill_null("").str.len_chars().alias("char_len"),
    pl.col("reviewText").fill_null("").str.count_matches(r"\S+").alias("word_len"),
])

pdf = df.to_pandas()
pdf["verified"] = pdf["verified"].fillna(False).astype(bool)
pdf["summary"] = pdf["summary"].fillna("").astype(str)
pdf["reviewText"] = pdf["reviewText"].fillna("").astype(str)
pdf["text"] = (pdf["summary"] + " " + pdf["reviewText"]).str.strip()

print("Shape:", pdf.shape)
print(pdf["sentiment"].value_counts(normalize=True).round(4))

## 3) Feature Analysis (No business insight write-up)

In [ ]:
feature_summary = (
    pdf.groupby('sentiment', as_index=False)
       .agg(
           n=('sentiment', 'size'),
           avg_word_len=('word_len', 'mean'),
           med_word_len=('word_len', 'median'),
           verified_rate=('verified', 'mean')
       )
)
feature_summary['verified_rate'] = feature_summary['verified_rate'].round(4)
feature_summary['avg_word_len'] = feature_summary['avg_word_len'].round(2)
print('Basic feature summary by sentiment:')
display(feature_summary)

In [ ]:
sample_for_features = pdf.sample(n=min(MAX_FEATURE_SAMPLE, len(pdf)), random_state=SEED)

vec = TfidfVectorizer(min_df=10, max_df=0.8, ngram_range=(1, 2), stop_words="english")
Xf = vec.fit_transform(sample_for_features["text"])
yf = sample_for_features["sentiment"]

coef_model = LogisticRegression(max_iter=300, random_state=SEED, class_weight="balanced")
coef_model.fit(Xf, yf)

feature_names = np.array(vec.get_feature_names_out())
classes = coef_model.classes_

top_terms = {}
for i, cls in enumerate(classes):
    coefs = coef_model.coef_[i]
    top_idx = np.argsort(coefs)[-20:][::-1]
    top_terms[cls] = feature_names[top_idx].tolist()

top_terms_df = pd.DataFrame({
    "negative_top_terms": pd.Series(top_terms.get("negative", [])),
    "neutral_top_terms": pd.Series(top_terms.get("neutral", [])),
    "positive_top_terms": pd.Series(top_terms.get("positive", [])),
})
display(top_terms_df.head(20))

## 4) Sentiment Modeling (Baseline vs Stronger Model)

In [ ]:
model_df = pdf[['text', 'sentiment']].dropna()
train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=SEED,
    stratify=model_df['sentiment']
)

print('Train size:', len(train_df), 'Test size:', len(test_df))

In [ ]:
pipelines = {
    'baseline_nb': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=5, max_df=0.9, ngram_range=(1, 1), stop_words='english')),
        ('clf', MultinomialNB())
    ]),
    'stronger_logreg': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=5, max_df=0.9, ngram_range=(1, 2), stop_words='english')),
        ('clf', LogisticRegression(max_iter=300, random_state=SEED, class_weight='balanced'))
    ]),
    'stronger_linearsvc': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=5, max_df=0.9, ngram_range=(1, 2), stop_words='english')),
        ('clf', LinearSVC(class_weight='balanced', random_state=SEED))
    ])
}

results = []
reports = {}
cms = {}

X_train, y_train = train_df['text'], train_df['sentiment']
X_test, y_test = test_df['text'], test_df['sentiment']

for name, pipe in pipelines.items():
    print(f'Training {name}...')
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, pred)
    macro_f1 = f1_score(y_test, pred, average='macro')
    weighted_f1 = f1_score(y_test, pred, average='weighted')

    results.append({
        'model': name,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    })

    reports[name] = classification_report(y_test, pred, output_dict=True)
    cms[name] = confusion_matrix(y_test, pred, labels=['negative', 'neutral', 'positive'])

results_df = pd.DataFrame(results).sort_values('macro_f1', ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_model = pipelines[best_model_name]
best_pred = best_model.predict(X_test)

print('Best model:', best_model_name)
print(classification_report(y_test, best_pred, digits=4))

cm_df = pd.DataFrame(
    cms[best_model_name],
    index=['true_negative', 'true_neutral', 'true_positive'],
    columns=['pred_negative', 'pred_neutral', 'pred_positive']
)
display(cm_df)

## 5) Topic Modeling on Negative Reviews (LDA vs NMF proxy)

In [ ]:
neg = pdf.loc[pdf["sentiment"] == "negative", "text"].dropna()
neg = neg[neg.str.len() > 30]

assert len(neg) > 0, "No negative reviews found after filtering."

neg_sample = neg.sample(n=min(MAX_NEG_SAMPLE, len(neg)), random_state=SEED)
print("Negative sample size for topic models:", len(neg_sample))

In [ ]:
n_topics = 12
count_vec = CountVectorizer(min_df=20, max_df=0.7, stop_words='english')
X_neg_count = count_vec.fit_transform(neg_sample)

lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=SEED,
    learning_method='batch',
    max_iter=20
)
lda.fit(X_neg_count)

tfidf_vec = TfidfVectorizer(min_df=20, max_df=0.7, stop_words='english')
X_neg_tfidf = tfidf_vec.fit_transform(neg_sample)

nmf = NMF(n_components=n_topics, random_state=SEED, init='nndsvda', max_iter=300)
nmf.fit(X_neg_tfidf)

print('LDA and NMF fitted.')

In [ ]:
def get_top_words(model, feature_names, n_top_words=12):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_idx = np.argsort(topic)[-n_top_words:][::-1]
        topics.append([feature_names[i] for i in top_idx])
    return topics

lda_topics = get_top_words(lda, count_vec.get_feature_names_out())
nmf_topics = get_top_words(nmf, tfidf_vec.get_feature_names_out())

lda_topics_df = pd.DataFrame({'topic_id': range(len(lda_topics)), 'top_words': [' | '.join(t) for t in lda_topics]})
nmf_topics_df = pd.DataFrame({'topic_id': range(len(nmf_topics)), 'top_words': [' | '.join(t) for t in nmf_topics]})

print('LDA Topics')
display(lda_topics_df)
print('NMF Topics')
display(nmf_topics_df)

In [ ]:
# Topic diversity as proportion of unique words among all top-k topic words.
def topic_diversity(topics, top_k=10):
    words = []
    for t in topics:
        words.extend(t[:top_k])
    return len(set(words)) / len(words)

div_lda = topic_diversity(lda_topics, top_k=10)
div_nmf = topic_diversity(nmf_topics, top_k=10)

topic_quality_df = pd.DataFrame([
    {'model': 'LDA', 'topic_diversity@10': div_lda},
    {'model': 'NMF', 'topic_diversity@10': div_nmf},
])
display(topic_quality_df)

## 6) Optional BERTopic Cell (Run only if package available)

In [ ]:
if not RUN_BERTOPIC:
    print("BERTopic skipped (RUN_BERTOPIC=False).")
else:
    try:
        from bertopic import BERTopic
        from sentence_transformers import SentenceTransformer

        bt_sample = neg_sample.sample(min(10000, len(neg_sample)), random_state=SEED).tolist()
        emb_model = SentenceTransformer("all-MiniLM-L6-v2")
        embeddings = emb_model.encode(bt_sample, show_progress_bar=True, batch_size=256)

        topic_model = BERTopic(
            min_topic_size=100,
            calculate_probabilities=False,
            verbose=True
        )
        bt_topics, _ = topic_model.fit_transform(bt_sample, embeddings)

        bt_info = topic_model.get_topic_info().head(20)
        display(bt_info)
        print("BERTopic completed.")
    except Exception as e:
        print("BERTopic not run:", e)
        print("This does not block the rest of the notebook.")

## 7) Error Analysis Artifact (for report completeness)

In [ ]:
err_df = test_df.copy()
err_df['pred'] = best_pred
err_df['is_error'] = err_df['pred'] != err_df['sentiment']

print('Total test errors:', int(err_df['is_error'].sum()))
display(err_df[err_df['is_error']].head(20))

## 8) Simple Dashboard Starter (Table-based, teammate can extend)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

meta = pl.read_csv(str(META_FILE), schema_overrides={'asin': pl.Utf8}).to_pandas()
dash_df = pdf[['asin', 'overall', 'sentiment', 'verified', 'word_len']].copy()
dash_df = dash_df.merge(meta[['asin', 'title', 'brand', 'avg_rating', 'review_count']], on='asin', how='left')

brand_options = ['ALL'] + sorted([b for b in dash_df['brand'].dropna().astype(str).unique()][:300])
sent_options = ['ALL', 'negative', 'neutral', 'positive']

brand_dd = widgets.Dropdown(options=brand_options, value='ALL', description='Brand:')
sent_dd = widgets.Dropdown(options=sent_options, value='ALL', description='Sentiment:')
topn = widgets.IntSlider(value=10, min=5, max=30, step=1, description='Top N:')
out = widgets.Output()

def render(_=None):
    with out:
        clear_output(wait=True)
        d = dash_df.copy()
        if brand_dd.value != 'ALL':
            d = d[d['brand'].astype(str) == brand_dd.value]
        if sent_dd.value != 'ALL':
            d = d[d['sentiment'] == sent_dd.value]

        if len(d) == 0:
            print('No rows for this filter.')
            return

        summary = d.groupby(['asin', 'title', 'brand'], as_index=False).agg(
            reviews=('overall', 'size'),
            avg_rating=('overall', 'mean'),
            neg_rate=('sentiment', lambda s: (s == 'negative').mean()),
            verified_rate=('verified', 'mean'),
            avg_word_len=('word_len', 'mean')
        )
        summary = summary.sort_values(['neg_rate', 'reviews'], ascending=[False, False]).head(topn.value)

        summary['avg_rating'] = summary['avg_rating'].round(3)
        summary['neg_rate'] = summary['neg_rate'].round(3)
        summary['verified_rate'] = summary['verified_rate'].round(3)
        summary['avg_word_len'] = summary['avg_word_len'].round(2)

        display(summary)

brand_dd.observe(render, names='value')
sent_dd.observe(render, names='value')
topn.observe(render, names='value')

display(widgets.HBox([brand_dd, sent_dd, topn]))
display(out)
render()

## 9) Rubric Checklist Artifact

In [ ]:
rubric_checklist = pd.DataFrame([
    ['Clarity/completeness', 'Data, labels, preprocessing documented in notebook sections 1-3'],
    ['Appropriate models', 'Baseline + stronger sentiment models; LDA + NMF topic comparison; optional BERTopic'],
    ['Coherence with hypotheses', 'Sentiment and complaint-theme modeling aligned with proposal objective'],
    ['Real-world usefulness', 'Product-level table outputs and model artifacts ready for downstream actioning'],
    ['Concise summarization support', 'Results tables and confusion matrices generated'],
    ['Advantages/limitations discussion support', 'Error analysis and optional BERTopic fallback documented']
], columns=['Rubric item', 'Notebook evidence'])
display(rubric_checklist)